# TIL — EDU-ORCH-001: Baseline vs Transformer Evidence Lab

Este experimento produz evidências comparáveis para a **Aula 13C — Model Routing, Orchestration e Utility**.

**Pergunta experimental:** em uma tarefa real de sentimento em português, qual é o trade-off entre:

- **TF-IDF + Multinomial Naive Bayes**;
- **DistilBERT multilingual**.

A comparação não procura “provar” que o Transformer vence. Ela mede **qualidade × latência × custo computacional** sob o mesmo conjunto de avaliação.

> **Política TIL:** valores reais, proxies e demonstrações permanecem explicitamente separados. Este notebook só grava `til-model-evidence.csv` quando a execução atende aos gates de evidência.


## 1. Conhecendo nossos dados — Dataset Card didático do TIL

### Origem

Usaremos o subconjunto **Olist / Polarity** do dataset preparado **Brazilian Portuguese Sentiment Analysis Datasets**.

- [Dataset preparado no Kaggle](https://www.kaggle.com/datasets/fredericods/ptbr-sentiment-analysis-datasets)
- [Dataset original — Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
- [Publicação científica — Sentiment Analysis on Brazilian Portuguese User Reviews](https://arxiv.org/abs/2112.05459)

### Contexto

O Olist original contém aproximadamente 100 mil pedidos realizados entre 2016 e 2018 e inclui avaliações de clientes. O dataset preparado organiza diferentes corpora de avaliações em português e fornece targets e partições reproduzíveis para comparação de modelos.

### Unidade de observação

Uma **review textual** associada a uma avaliação de 1 a 5 estrelas.

### Campos utilizados

| Campo | Papel no experimento |
|---|---|
| `review_text` | entrada textual comum aos dois sistemas |
| `polarity` | target binário |
| `rating` | auditoria da transformação estrela → sentimento |
| `kfold_polarity` | definição oficial de treino/validação/teste |

### Target e transformação

A preparação define:

```text
1 ou 2 estrelas → 0 = negativo
3 estrelas      → removida do target de polaridade
4 ou 5 estrelas → 1 = positivo
```

Essa decisão transforma um problema ordinal de cinco níveis em um problema binário e remove os casos intermediários. Consequências:

1. a tarefa deixa de representar a escala completa de satisfação;
2. exemplos ambíguos/intermediários são retirados;
3. a fronteira de decisão tende a ficar mais nítida;
4. resultados não devem ser generalizados como se o modelo previsse ratings de 1 a 5.

### Distribuição das classes

A documentação do dataset reporta, para Olist/Polarity, aproximadamente **30% negativos** e **70% positivos**. Por isso usamos **F1 macro** como métrica principal e accuracy apenas como complemento.

### Preparação e split

A fonte preparada define 10 folds estratificados para polaridade. Neste experimento:

```text
folds 1–8 → treino
fold 9     → validação
fold 10    → teste
```

O conjunto de teste é idêntico para os dois sistemas.

### Limitações

- reviews do domínio de e-commerce;
- textos curtos;
- desbalanceamento entre classes;
- sentimento é derivado do rating, não anotado independentemente;
- reviews de 3 estrelas são excluídas do target binário;
- resultados não representam automaticamente outros domínios de português.

### Licença e governança

O **dataset Olist original** é publicado no Kaggle sob **CC BY-NC-SA 4.0**. A página do **dataset preparado** informa atualmente licença **Unknown**. O TIL registra ambas as informações e **não presume** que a licença do artefato derivado seja automaticamente a mesma do dataset original.

### Papel no experimento

O dataset serve como base comum para comparar duas arquiteturas de classificação textual e, depois, alimentar decisões de **Model Selection → Model Routing → Model Orchestration → Compound AI Systems**.


## 2. Contrato experimental

Antes de observar resultados, fixamos:

- **quality principal:** `f1_macro`;
- **quality complementar:** `accuracy`;
- **split:** folds 1–8 / 9 / 10;
- **input comum:** `review_text`;
- **baseline:** TF-IDF + MultinomialNB;
- **Transformer:** `distilbert-base-multilingual-cased`;
- **latência:** end-to-end por exemplo, `batch_size=1`, com média, p50 e p95;
- **proxy de custo:** segundos de runtime por 1.000 inferências em lote;
- **evidência:** somente exportada quando dataset real e ambos os sistemas forem efetivamente medidos.

Não há custo monetário inferido neste experimento.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
RUN_MODE = "EVIDENCE" if IS_KAGGLE else "SMOKE"

print("RUN_MODE:", RUN_MODE)
print("Python:", platform.python_version())
print("Platform:", platform.platform())


## 3. Descoberta determinística dos recursos

No Kaggle, o notebook espera:

- o dataset `fredericods/ptbr-sentiment-analysis-datasets` anexado;
- o modelo `goddiao/distilbert-base-multilingual-cased`, PyTorch/default/version 1, anexado;
- Internet OFF.

Localmente, a ausência desses recursos não gera evidência falsa: o notebook encerra as etapas de medição de forma controlada.


In [ ]:
DATASET_ROOT = Path("/kaggle/input")
MODEL_DIR = Path(
    "/kaggle/input/models/goddiao/distilbert-base-multilingual-cased/"
    "pytorch/default/1/distilbert-base-multilingual-cased"
)

def find_olist_csv(root: Path) -> Path | None:
    if not root.exists():
        return None
    candidates = []
    for p in root.rglob("*.csv"):
        name = p.name.lower()
        if "olist" in name:
            candidates.append(p)
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda p: (p.name.lower() != "olist.csv", len(str(p))))
    return candidates[0]

OLIST_CSV = find_olist_csv(DATASET_ROOT) if IS_KAGGLE else None
print("OLIST_CSV:", OLIST_CSV)
print("MODEL_DIR exists:", MODEL_DIR.exists())

RESOURCES_READY = bool(OLIST_CSV and OLIST_CSV.exists() and MODEL_DIR.exists())
print("RESOURCES_READY:", RESOURCES_READY)


In [ ]:
EXPECTED_COLUMNS = {
    "review_text", "polarity", "rating", "kfold_polarity"
}

if RESOURCES_READY:
    df = pd.read_csv(OLIST_CSV)
    missing = EXPECTED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Dataset incompatível. Colunas ausentes: {sorted(missing)}")

    work = df.loc[df["polarity"].notna(), [
        "review_text", "polarity", "rating", "kfold_polarity"
    ]].copy()
    work["review_text"] = work["review_text"].astype(str)
    work["polarity"] = work["polarity"].astype(int)
    work["kfold_polarity"] = work["kfold_polarity"].astype(int)

    train_df = work[work["kfold_polarity"].between(1, 8)].reset_index(drop=True)
    val_df = work[work["kfold_polarity"].eq(9)].reset_index(drop=True)
    test_df = work[work["kfold_polarity"].eq(10)].reset_index(drop=True)

    print("rows total polarity:", len(work))
    print("train:", len(train_df), "validation:", len(val_df), "test:", len(test_df))
    print("\nClass distribution:")
    print(work["polarity"].value_counts(normalize=True).sort_index())
else:
    df = work = train_df = val_df = test_df = None
    print("SMOKE: dataset/model Kaggle não disponíveis; medições reais serão ignoradas.")


## 4. Gate de integridade do target

Além de confiar na documentação, o notebook verifica nos dados carregados se:

- ratings 1–2 correspondem a polaridade 0;
- ratings 4–5 correspondem a polaridade 1;
- ratings 3 não aparecem entre exemplos com target de polaridade válido.

Se o gate falhar, o experimento é interrompido antes de treinar modelos.


In [ ]:
if RESOURCES_READY:
    bad_neg = work.loc[work["rating"].isin([1, 2]) & work["polarity"].ne(0)]
    bad_pos = work.loc[work["rating"].isin([4, 5]) & work["polarity"].ne(1)]
    rating3 = work.loc[work["rating"].eq(3)]

    assert bad_neg.empty, "Gate falhou: rating 1–2 não mapeia integralmente para polarity=0."
    assert bad_pos.empty, "Gate falhou: rating 4–5 não mapeia integralmente para polarity=1."
    assert rating3.empty, "Gate falhou: ratings 3 aparecem no target binário."

    print("Target integrity gate: PASS")
else:
    print("Target integrity gate: SKIPPED (SMOKE)")


## 5. Sistema A — TF-IDF + Multinomial Naive Bayes

O baseline usa uma implementação direta, previamente ensinada no TIL. Não há seleção retrospectiva de vários classificadores para escolher o resultado mais favorável.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

baseline_metrics = None
baseline_pipeline = None

if RESOURCES_READY:
    from sklearn.pipeline import Pipeline

    baseline_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True,
        )),
        ("nb", MultinomialNB(alpha=1.0)),
    ])

    baseline_pipeline.fit(train_df["review_text"], train_df["polarity"])
    baseline_pred = baseline_pipeline.predict(test_df["review_text"])

    baseline_metrics = {
        "f1_macro": float(f1_score(test_df["polarity"], baseline_pred, average="macro")),
        "accuracy": float(accuracy_score(test_df["polarity"], baseline_pred)),
    }
    print(baseline_metrics)
else:
    print("Baseline training: SKIPPED (SMOKE)")


## 6. Sistema B — DistilBERT multilingual

Recurso homologado no TIL:

```text
goddiao/distilbert-base-multilingual-cased
framework: PyTorch
variation: default
version: 1
```

A cabeça de classificação é nova para esta tarefa. A seed é fixada **antes** da criação do modelo.


In [ ]:
transformer_metrics = None
transformer_model = None
tokenizer = None
device = None

if RESOURCES_READY:
    import torch
    from torch.utils.data import Dataset
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    os.environ["PYTHONHASHSEED"] = str(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    set_seed(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    tokenizer = AutoTokenizer.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
    )

    class ReviewDataset(Dataset):
        def __init__(self, frame, tokenizer, max_length=128):
            self.texts = frame["review_text"].tolist()
            self.labels = frame["polarity"].astype(int).tolist()
            self.tokenizer = tokenizer
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            item = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
            )
            item = {k: v.squeeze(0) for k, v in item.items()}
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    train_ds = ReviewDataset(train_df, tokenizer)
    val_ds = ReviewDataset(val_df, tokenizer)
    test_ds = ReviewDataset(test_df, tokenizer)

    id2label = {0: "negative", 1: "positive"}
    label2id = {"negative": 0, "positive": 1}

    set_seed(SEED)
    transformer_model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "f1_macro": f1_score(labels, preds, average="macro"),
            "accuracy": accuracy_score(labels, preds),
        }

    args = TrainingArguments(
        output_dir="/kaggle/working/edu-orch-001-distilbert",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        report_to=[],
        seed=SEED,
        data_seed=SEED,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=transformer_model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    test_result = trainer.predict(test_ds)
    transformer_metrics = {
        "f1_macro": float(test_result.metrics["test_f1_macro"]),
        "accuracy": float(test_result.metrics["test_accuracy"]),
    }
    print(transformer_metrics)
else:
    print("Transformer training: SKIPPED (SMOKE)")


## 7. Benchmark de latência

O benchmark mede o pipeline **end-to-end** em `batch_size=1`.

- baseline: texto → TF-IDF → NB → predição;
- Transformer: texto → tokenizer → device → forward → argmax.

Usamos um subconjunto fixo do test set para manter a medição reproduzível.


In [ ]:
LATENCY_SAMPLE_SIZE = 256
WARMUP = 20

def summarize_ms(values):
    a = np.asarray(values, dtype=float)
    return {
        "latency_ms": float(a.mean()),
        "latency_p50_ms": float(np.percentile(a, 50)),
        "latency_p95_ms": float(np.percentile(a, 95)),
    }

baseline_latency = transformer_latency = None

if RESOURCES_READY:
    latency_df = test_df.sample(
        n=min(LATENCY_SAMPLE_SIZE, len(test_df)),
        random_state=SEED
    ).reset_index(drop=True)

    # Baseline warm-up
    for text in latency_df["review_text"].iloc[:min(WARMUP, len(latency_df))]:
        baseline_pipeline.predict([text])

    b_times = []
    for text in latency_df["review_text"]:
        t0 = time.perf_counter()
        baseline_pipeline.predict([text])
        b_times.append((time.perf_counter() - t0) * 1000)
    baseline_latency = summarize_ms(b_times)

    import torch
    transformer_model.eval()
    transformer_model.to(device)

    def transformer_predict_one(text):
        encoded = tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        if device.type == "cuda":
            torch.cuda.synchronize()
        with torch.inference_mode():
            out = transformer_model(**encoded)
        if device.type == "cuda":
            torch.cuda.synchronize()
        return int(out.logits.argmax(dim=-1).item())

    for text in latency_df["review_text"].iloc[:min(WARMUP, len(latency_df))]:
        transformer_predict_one(text)

    t_times = []
    for text in latency_df["review_text"]:
        t0 = time.perf_counter()
        transformer_predict_one(text)
        t_times.append((time.perf_counter() - t0) * 1000)
    transformer_latency = summarize_ms(t_times)

    print("baseline latency:", baseline_latency)
    print("transformer latency:", transformer_latency)
else:
    print("Latency benchmark: SKIPPED (SMOKE)")


## 8. Proxy de custo computacional

Não usamos preço monetário.

O campo `cost_per_1000` representa:

> **segundos de runtime medidos para 1.000 inferências em lote**, normalizados a partir de uma amostra fixa.

É um **proxy computacional**, não moeda.

A medição é separada do benchmark de latência: latência usa `batch_size=1`; custo proxy usa processamento em lote.


In [ ]:
COST_SAMPLE_SIZE = 1000
TRANSFORMER_BATCH_SIZE = 32

baseline_cost = transformer_cost = None

if RESOURCES_READY:
    cost_df = test_df.sample(
        n=min(COST_SAMPLE_SIZE, len(test_df)),
        random_state=SEED + 1
    ).reset_index(drop=True)
    n_cost = len(cost_df)

    # Baseline throughput
    t0 = time.perf_counter()
    _ = baseline_pipeline.predict(cost_df["review_text"].tolist())
    elapsed_b = time.perf_counter() - t0
    baseline_cost = elapsed_b * (1000.0 / n_cost)

    # Transformer throughput
    import torch
    t0 = time.perf_counter()
    with torch.inference_mode():
        texts = cost_df["review_text"].tolist()
        for start in range(0, len(texts), TRANSFORMER_BATCH_SIZE):
            batch_texts = texts[start:start + TRANSFORMER_BATCH_SIZE]
            encoded = tokenizer(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=128,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            if device.type == "cuda":
                torch.cuda.synchronize()
            _ = transformer_model(**encoded).logits.argmax(dim=-1)
            if device.type == "cuda":
                torch.cuda.synchronize()
    elapsed_t = time.perf_counter() - t0
    transformer_cost = elapsed_t * (1000.0 / n_cost)

    print("baseline runtime_seconds_per_1000:", baseline_cost)
    print("transformer runtime_seconds_per_1000:", transformer_cost)
else:
    print("Cost proxy benchmark: SKIPPED (SMOKE)")


## 9. Evidência consolidada

Somente medições reais entram nesta tabela. O notebook não preenche linhas demonstrativas.


In [ ]:
evidence_df = pd.DataFrame()

if RESOURCES_READY:
    import sklearn
    import torch
    import transformers

    measured_at = datetime.now(timezone.utc).isoformat()
    hardware = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else platform.processor() or platform.machine()
    )
    dataset_desc = "Olist/Polarity; folds 1-8 train, 9 validation, 10 test"

    rows = [
        {
            "system": "tfidf_multinomial_nb",
            "quality": baseline_metrics["f1_macro"],
            "quality_metric": "f1_macro",
            "accuracy": baseline_metrics["accuracy"],
            "cost_per_1000": baseline_cost,
            "cost_unit": "runtime_seconds_per_1000",
            "cost_method": "measured_batch_runtime_proxy",
            "latency_ms": baseline_latency["latency_ms"],
            "latency_p50_ms": baseline_latency["latency_p50_ms"],
            "latency_p95_ms": baseline_latency["latency_p95_ms"],
            "source": "EDU-ORCH-001",
            "measured_at": measured_at,
            "evidence_status": "measured",
            "dataset": dataset_desc,
            "hardware": hardware,
            "sample_size": len(test_df),
            "model_version": f"scikit-learn={sklearn.__version__}; MultinomialNB(alpha=1.0)",
            "notes": "Cost is computational proxy, not monetary price.",
        },
        {
            "system": "distilbert_multilingual",
            "quality": transformer_metrics["f1_macro"],
            "quality_metric": "f1_macro",
            "accuracy": transformer_metrics["accuracy"],
            "cost_per_1000": transformer_cost,
            "cost_unit": "runtime_seconds_per_1000",
            "cost_method": "measured_batch_runtime_proxy",
            "latency_ms": transformer_latency["latency_ms"],
            "latency_p50_ms": transformer_latency["latency_p50_ms"],
            "latency_p95_ms": transformer_latency["latency_p95_ms"],
            "source": "EDU-ORCH-001",
            "measured_at": measured_at,
            "evidence_status": "measured",
            "dataset": dataset_desc,
            "hardware": hardware,
            "sample_size": len(test_df),
            "model_version": (
                "goddiao/distilbert-base-multilingual-cased/"
                f"pytorch/default/1; transformers={transformers.__version__}"
            ),
            "notes": "Fine-tuned 2 epochs; cost is computational proxy, not monetary price.",
        },
    ]
    evidence_df = pd.DataFrame(rows)
    display(evidence_df)
else:
    print("Evidence table: EMPTY in SMOKE mode.")


## 10. Export do contrato TIL

O CSV resumido permanece compatível com a Aula 13C porque preserva os quatro campos mínimos:

```text
system
quality
cost_per_1000
latency_ms
```

Campos adicionais mantêm contexto e auditabilidade.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/model-evidence") if IS_KAGGLE else Path("data/model-evidence")
OUTPUT_CSV = OUTPUT_DIR / "til-model-evidence.csv"

if RESOURCES_READY and len(evidence_df) >= 2:
    required = {"system", "quality", "cost_per_1000", "latency_ms"}
    assert required.issubset(evidence_df.columns)
    assert evidence_df["quality"].between(0, 1).all()
    assert evidence_df["cost_per_1000"].ge(0).all()
    assert evidence_df["latency_ms"].ge(0).all()
    assert evidence_df["evidence_status"].eq("measured").all()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    evidence_df.to_csv(OUTPUT_CSV, index=False)
    print("Evidence export: PASS")
    print("CSV:", OUTPUT_CSV)
else:
    print("Evidence export: SKIPPED — execução não atende ao gate de evidência.")


## 11. Interpretação: do Model Selection ao Compound AI System

O experimento responde primeiro a uma pergunta de **Model Selection**:

> qual sistema apresenta o trade-off mais adequado sob as condições medidas?

A Aula 13C reutiliza essas linhas para avançar:

```text
Model Selection
      ↓
Model Routing
      ↓
Model Orchestration
      ↓
Compound AI Systems
```

Um modelo com maior F1 macro pode ter latência e custo computacional superiores. Um baseline simples pode continuar sendo a melhor escolha para parte das requisições. Essa tensão é justamente o fundamento de routing e cascades.


In [ ]:
manifest = {
    "experiment": "EDU-ORCH-001",
    "run_mode": RUN_MODE,
    "resources_ready": RESOURCES_READY,
    "dataset_path": str(OLIST_CSV) if OLIST_CSV else None,
    "model_dir": str(MODEL_DIR),
    "seed": SEED,
    "quality_metric": "f1_macro",
    "cost_unit": "runtime_seconds_per_1000",
    "latency_sample_size": LATENCY_SAMPLE_SIZE,
    "cost_sample_size": COST_SAMPLE_SIZE,
    "measured_at": datetime.now(timezone.utc).isoformat(),
}
print(json.dumps(manifest, indent=2, ensure_ascii=False))

if RUN_MODE == "SMOKE":
    print("\nHEADLESS SMOKE: PASS — nenhuma evidência real foi fabricada.")
elif RESOURCES_READY:
    print("\nEVIDENCE RUN: COMPLETE")
